In [ ]:
# Import libraries
from transformers import AutoTokenizer, LlamaForCausalLM
import torch
from huggingface_hub import login
import os
import openpyxl
import ast
import json
import time
import numpy as np
import re
import torch
print("Done importing packages")

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# Login to huggingface
path_to_hf_key = 'pat/to/you/huggingface/key'
with open(path_to_hf_key, 'r') as f:
    token = f.read().strip()
login(token=token, add_to_git_credential=True)
print("Logged in")

In [ ]:
# Select model
model_id = "meta-llama/Llama-3.2-1B-Instruct"


In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id,local_files_only = True)
#Define end tokens
terminators = [
            tokenizer.eos_token_id, #retrieves end of sequence ID
            tokenizer.convert_tokens_to_ids("<|eot_id|>"),
            #converts this custom end of turn token to system eos
        ]

print("tokenizer loaded")

# Load model
model = LlamaForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            #This parameter specifies the data type to use for the model's weights.
            # torch.bfloat16 is a 16-bit floating-point format that offers a good trade-off between memory usage and model accuracy.
            device_map="auto",
            local_files_only = True
        )

# Check if the model's parameters are on the GPU
if next(model.parameters()).is_cuda:
    print("Model is on GPU.")
else:
    print("Model is on CPU.")

In [ ]:
def run_model(new_prompt,dlg_hist,max_new_tokens,do_sample,temperature,top_p):
    #dlg_hist needs to be list type
    prompt =  dlg_hist.copy() #Creates new list, but content links to same dictionaries
    prompt.append(new_prompt) #New prompt (I must supply it in prompt format)
    start_time = time.time()
    input_ids = tokenizer.apply_chat_template(
            prompt,
            add_generation_prompt=True, #Adds special formatting of input
            return_tensors="pt", #Returns Pytorch tensors
            padding=False, #Padding to make all sequences equally long, with short ones filled with padding
        ).to(model.device) #Moves the input tensors to where model is (GPU)
    outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            pad_token_id =128256,
            do_sample=do_sample, 
            temperature=temperature,
            top_p=top_p
        )
    outputs.shape #Returns tuple of (number of sequences, length of sequences)

    response = outputs[0][input_ids.shape[-1]:] #Takes first output sequence and removes the bit that is just a copy of the input, so we get just response
    decoded_response = tokenizer.decode(response, skip_special_tokens=True)
    inference_time = time.time() - start_time
    return decoded_response, inference_time

In [ ]:
# Content of the query point text file:
# Context
# Prompt
# Task
# Ground truth response (for reference, not to be given to model)

# Function to read query points, prompt model, and fill Excel sheet with responses and metadata
def fill_excel_from_text(txt_filename, excel_path, max_new_tokens, do_sample, temperature, top_p, approach, system_prompt_file,context_window):
    # Read the prompt and dialogue history from the text file
    with open(txt_filename, 'r') as file:
        lines = file.readlines()

    dialogue_history_to_append = [ast.literal_eval(line.strip()) for line in lines[:context_window]]  # First context window lines are dialogue history (last index in range not included). LIST format
    new_prompt = ast.literal_eval(lines[context_window].strip())  # Final line of context window the prompt (context window +1), but in [] we put one number before (Python indexes), so "context window"
    #strip() removes trailing whitespace

	# Extract task instructions line (the line after the prompt)
    task_instructions_line = ast.literal_eval(lines[context_window+2].strip())

    # Extract system prompt if one is provided:
    if system_prompt_file != None:
        with open(system_prompt_file, 'r') as system_prompt_file:
            system_lines_content = system_prompt_file.read()
        system_lines_as_dict = json.loads(system_lines_content) #So that system prompt is not converted to string
        task_instructions_content_string = task_instructions_line["task_instructions"]
        system_lines_as_dict["content"] = system_lines_as_dict["content"] + task_instructions_content_string
        # Add dialogue history after system prompt
        dialogue_history = []
        dialogue_history.append(system_lines_as_dict)
        for history_line in dialogue_history_to_append:
            dialogue_history.append(history_line)
    else:
        #Use dialogue history as is, without adding system prompt at beginning
        dialogue_history = []
        for history_line in dialogue_history_to_append:
            dialogue_history.append(history_line)
    
    # Extract Dialogue and QP type from the filename (if the filename contains information about special query types
    base_filename = os.path.basename(txt_filename)
    dialogue_field = base_filename.split('-')[0]  # Assuming "d\dx\d\d - QPblabla" format
    qp_type = " - ".join(base_filename.split(' - ')[1:]).replace('.txt', '')  # Everything else is QP

	# Define model name column for the output Excel
    model_name = model_id

	# ---------------------------------------------------------------------------------------------------------------------
    # Call the LLM model 3 times and record responses + inference times (we do 3 runs in case there are any blank responses)
    responses = []
    total_inference_time = 0
    for _ in range(3):
        response, inference_time = run_model(new_prompt, dialogue_history, max_new_tokens, do_sample, temperature, top_p)
        responses.append(response)
        total_inference_time += inference_time

    avg_inference_time = total_inference_time / 3

	# ---------------------------------------------------------------------------------------------------------------------
    # Load the Excel file and fill it out
    # Column names to add as the header if the fiel does not exist
    headers = [
    "Model", "Dialogue history", "Prompt", "Response 1", "Response 2", "Response 3", 
    "Do sample", "Temperature", "Top p", "Max new tokens", "Thoughts", 
    "Dialogue", "QP type", "Inference time (seconds)", "Approach"
    ]
    # Check if the file exists
    if not os.path.exists(excel_path):
        # If the file doesn't exist, create a new workbook and save it
        wb = openpyxl.Workbook()  # Create a new workbook
        ws = wb.active  # Get the active sheet
        # Add the headers to the first row
        for col_num, header in enumerate(headers, 1):
            ws.cell(row=1, column=col_num).value = header
        wb.save(excel_path)  # Save it to the specified path
    wb = openpyxl.load_workbook(excel_path)
    sheet = wb.active

    # Find the next empty row to add data
    next_row = sheet.max_row + 1

    # Fill in the columns
    sheet[f"A{next_row}"] = model_name                     # Model
    sheet[f"B{next_row}"] = str(dialogue_history)          # Dialogue history
    sheet[f"C{next_row}"] = str(new_prompt)                # Prompt
    sheet[f"D{next_row}"] = responses[0]                   # Response 1
    sheet[f"E{next_row}"] = responses[1]                   # Response 2
    sheet[f"F{next_row}"] = responses[2]                   # Response 3
    sheet[f"G{next_row}"] = do_sample                      # Do sample
    sheet[f"H{next_row}"] = temperature                    # Temperature
    sheet[f"I{next_row}"] = top_p                          # Top p
    sheet[f"J{next_row}"] = max_new_tokens                 # Max new tokens
    # Leave Thoughts (column K) empty to add manually during analysis
    sheet[f"L{next_row}"] = dialogue_field                 # Dialogue No
    sheet[f"M{next_row}"] = qp_type                        # QP type
    sheet[f"N{next_row}"] = avg_inference_time             # Inference time
    sheet[f"O{next_row}"] = approach                       # Approach to prompting

    

    # Save the updated Excel file
    wb.save(excel_path)
    print(f"Row {next_row} added to {excel_path}")


In [ ]:
# ----------------------------------------------------
# RUN THE PIPELINE
# ----------------------------------------------------
# Iterate through files in directory and find QP files in the format \d\d\dx\d\d\d-QP_\d+.txt
# Use that info for text_filename variable and to create excel_filename variable
# Run fill_excel_from text with those variables

def find_qp_files_and_process(directory,context_window, max_new_tokens, do_sample, temperature, top_p, approach, system_prompt_file):
    # Define the regex pattern to find QP files
    pattern = re.compile(r"\d\d\dx\d\d\d-QP-CONTEXT\d+-with GT_\d+.*.txt")

    # Iterate through the files in the specified directory
    for filename in os.listdir(directory):
        # Check that the filename corresponds to the query point filename style
        if "txt" in filename:
            if pattern.match(filename):
                print(filename)
                text_filename = os.path.join(directory, filename)
                # Generate the corresponding Excel filename (replace .txt with .xlsx)
                excel_filename = os.path.splitext(text_filename)[0] + 'OUTPUT.xlsx'


                # Call the `fill_excel_from_text` function with desired parameters
                fill_excel_from_text(text_filename, excel_filename, max_new_tokens, do_sample, temperature, top_p, approach, system_prompt_file,context_window)
                
				# Commands to avoid memory issues
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()


In [ ]:
# Run the main function to find and process QP files
directory = 'path/to/your/QP/files'
context_window = 50 # Replace with your desired context window size
max_new_tokens = 800 # Replace with your desired max new tokens
do_sample = True # Replace with your desired sampling setting (True or False)
temperature = 0.6 # Replace with your desired temperature setting
top_p = 0.9 # Replace with your desired top_p setting
approach = "name of you prompting approach, e.g. Prompt6-with-system-prompt4"
system_prompt_file = 'path/to/your/system/prompt.txt'

find_qp_files_and_process(directory, context_window, max_new_tokens, do_sample, temperature, top_p, approach, system_prompt_file)